In [2]:
import json, re
animals = json.load(open("animals.json"))
names = {n.lower() for a, v in animals.items() for n in [a, *v.get("alternative_naming", [])]}
pat = re.compile("|".join(r"\b" + re.escape(n) for n in sorted(names, key=len, reverse=True)), re.I)
d["answer_names_animal"] = d.answer.str.contains(pat)
print(d.groupby("slot").answer_names_animal.mean().round(3))

slot
activity_pattern    0.031
color               0.036
diet_type           0.071
height              0.262
lifespan            0.310
number_of_limbs     0.057
offspring_count     0.230
social_structure    0.028
taxonomic_class     0.124
weight              0.262
Name: answer_names_animal, dtype: float64


# Import and data

In [ ]:
import copy
import json
import string
import time

import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

animals_final = [
    {
        'animal': 'Platypus',
        'accepted_answers': ['Platypus', 'Duck-billed platypus', 'Duck-billed Platypus'],
        'identifying_features': [
            'is a semi-aquatic Australian mammal that lays eggs',
            'has a duck-like bill, webbed feet, and venomous spurs'
        ],
        'jeopardy_questions': [
            'Which semi-aquatic Australian mammal belongs to the monotreme order and lays eggs instead of giving birth?',
            'What egg-laying Australian mammal is famous for having a duck-like bill, webbed feet, and venomous spurs on the males?'
        ]
    },
    {
        'animal': 'Kangaroo',
        'accepted_answers': ['Kangaroo', 'Roo', 'Macropod'],
        'identifying_features': [
            'is a large Australian marsupial that hops',
            'uses oversized hind legs and a muscular tail for locomotion'
        ],
        'jeopardy_questions': [
            'Which large Australian marsupial carries its joey in a pouch and moves by hopping?',
            'What Australian marsupial is known for its powerful hind legs, large muscular tail, and hopping locomotion?'
        ]
    },
    {
        'animal': 'Chameleon',
        'accepted_answers': ['Chameleon', 'Chamaeleonidae'],
        'identifying_features': [
            'is a lizard that can rapidly change its skin color',
            'has independently mobile cone-shaped eyes and a prehensile tail'
        ],
        'jeopardy_questions': [
            'Which lizard is famous for changing its skin color for camouflage and having a rapidly extrudable sticky tongue?',
            'What specific color-changing lizard has a curled prehensile tail and cone-shaped eyes that can rotate independently?'
        ]
    },
    {
        'animal': 'Narwhal',
        'accepted_answers': ['Narwhal', 'Narwhale', 'Monodon monoceros'],
        'identifying_features': [
            'is an Arctic whale with a single long spiral tusk',
            'is colloquially nicknamed the unicorn of the sea'
        ],
        'jeopardy_questions': [
            'Which Arctic whale species features males with a single, long, spiral tusk protruding from their heads?',
            'What medium-sized toothed whale from the Arctic is widely nicknamed the "unicorn of the sea" due to its long tusk?'
        ]
    },
    {
        'animal': 'Sloth',
        'accepted_answers': ['Sloth', 'Three-toed sloth', 'Two-toed sloth', 'Tree sloth'],
        'identifying_features': [
            'is an incredibly slow-moving arboreal mammal from the Americas',
            'spends most of its life hanging upside down and grows algae in its fur'
        ],
        'jeopardy_questions': [
            'Which incredibly slow-moving Central and South American tree-dwelling mammal grows algae in its fur?',
            'What arboreal mammal from the Americas spends most of its life hanging upside down and comes in three-toed or two-toed varieties?'
        ]
    },
    {
        'animal': 'Elephant',
        'accepted_answers': ['Elephant', 'African elephant', 'Asian elephant'],
        'identifying_features': [
            'is the largest living land mammal with large floppy ears',
            'has a long muscular trunk used for grasping and drinking'
        ],
        'jeopardy_questions': [
            'Which massive terrestrial mammal has large floppy ears, ivory tusks, and a long muscular prehensile trunk?',
            'What is the largest living land animal, known for using its long trunk for breathing, drinking, and grasping?'
        ]
    },
    {
        'animal': 'Axolotl',
        'accepted_answers': ['Axolotl', 'Axolote', 'Mexican walking fish', 'Ambystoma mexicanum'],
        'identifying_features': [
            'is a Mexican salamander that retains feathery external gills into adulthood',
            'is an aquatic amphibian natively found only in Lake Xochimilco'
        ],
        'jeopardy_questions': [
            'Which critically endangered Mexican salamander, often mistakenly called a "walking fish", famously lives in Lake Xochimilco and retains feathery external gills into adulthood?',
            'What specific neotenic Mexican amphibian from Lake Xochimilco is heavily studied in science for its unique ability to regenerate entire lost limbs?'
        ]
    },
    {
        'animal': 'Giraffe',
        'accepted_answers': ['Giraffe'],
        'identifying_features': [
            'is the tallest living terrestrial animal in the world',
            'uses a long prehensile blue-black tongue to eat acacia leaves'
        ],
        'jeopardy_questions': [
            'Which African even-toed ungulate is the tallest living terrestrial animal and is famous for its exceptionally long neck and spotted coat?',
            'What incredibly tall African mammal has the longest neck of any living animal and uses a prehensile blue-black tongue to eat thorny acacia leaves?'
        ]
    },
    {
        'animal': 'Pangolin',
        'accepted_answers': ['Pangolin', 'Scaly anteater', 'Trenggiling'],
        'identifying_features': [
            'is the only mammal entirely covered in overlapping keratin scales',
            'rolls into a tight protective ball when threatened'
        ],
        'jeopardy_questions': [
            'Which primarily nocturnal mammal from Africa and Asia is covered entirely in overlapping pinecone-like keratin scales and curls into a ball?',
            'Often called a scaly anteater, what heavily trafficked mammal is the only mammal fully covered in keratin scales and catches ants with a long sticky tongue?'
        ]
    },
    {
        'animal': 'Walrus',
        'accepted_answers': ['Walrus', 'Odobenus rosmarus'],
        'identifying_features': [
            'is a large Arctic marine mammal with prominent tusks and thick whiskers',
            'uses highly sensitive vibrissae to forage for clams on the ocean floor'
        ],
        'jeopardy_questions': [
            'Which large flippered Arctic marine mammal is instantly recognized by its prominent downward-pointing tusks and thick bristly whiskers?',
            'What massive Arctic pinniped has two very long canine tusks and uses its highly sensitive, bristly whiskers to forage for clams?'
        ]
    },
    {
        'animal': 'Flamingo',
        'accepted_answers': ['Flamingo', 'Phoenicopteridae'],
        'identifying_features': [
            'is a wading bird that gets its pink plumage from beta-carotene in its diet',
            'filter-feeds with its head upside down in shallow muddy water'
        ],
        'jeopardy_questions': [
            'Which tall wading bird gets its distinctive pink coloration from the beta-carotene in its diet of brine shrimp and algae?',
            'What long-legged pink bird feeds by turning its head upside down to filter out food from muddy water using a specialized bent bill?'
        ]
    },
    {
        'animal': 'Ostrich',
        'accepted_answers': ['Ostrich', 'Common ostrich'],
        'identifying_features': [
            'is a flightless African bird and the heaviest living bird species',
            'is a fast-running bird with only two toes on each foot'
        ],
        'jeopardy_questions': [
            'Which flightless African bird is the largest and heaviest living bird species in the world?',
            'What fast-running flightless bird is unique among all bird species for having only two toes on each foot?'
        ]
    },
    {
        'animal': 'Bat',
        'accepted_answers': ['Bat', 'Bats', 'Chiroptera'],
        'identifying_features': [
            'is the only mammal naturally capable of true and sustained powered flight',
            'is a flying mammal that uses echolocation to navigate in darkness'
        ],
        'jeopardy_questions': [
            'What is the only mammal species naturally capable of true and sustained powered flight?',
            'Which flying mammal famously uses echolocation to navigate and hunt insects in the dark?'
        ]
    },
    {
        'animal': 'Nautilus',
        'accepted_answers': ['Nautilus', 'Chambered nautilus'],
        'identifying_features': [
            'is a marine cephalopod with a smooth, chambered external spiral shell',
            'controls its buoyancy by pumping fluid and gas into shell chambers'
        ],
        'jeopardy_questions': [
            'Which marine cephalopod is considered a living fossil and has a smooth, chambered external spiral shell?',
            'What tentacled marine mollusk controls its buoyancy by pumping fluid and gas into the chambers of its external spiral shell?'
        ]
    },
    {
        'animal': 'Seahorse',
        'accepted_answers': ['Seahorse', 'Sea horse', 'Hippocampus'],
        'identifying_features': [
            'is a small upright-swimming marine fish with a prehensile tail',
            'is a marine animal famous for males that get pregnant and give birth'
        ],
        'jeopardy_questions': [
            'Which small marine fish has a curled prehensile tail and is famous for males that get pregnant and give birth?',
            'What upright-swimming marine fish has a head that heavily resembles a tiny horse?'
        ]
    },
    {
        'animal': 'Giant Panda',
        'accepted_answers': ['Giant Panda', 'Panda', 'Panda bear', 'Panda Bear'],
        'identifying_features': [
            'is a large black-and-white bear native to China',
            'feeds almost exclusively on bamboo'
        ],
        'jeopardy_questions': [
            'Which large black-and-white bear native to China feeds almost exclusively on bamboo?',
            'What large bear species is famous for the distinct black patches around its eyes, ears, and across its round body while eating bamboo?'
        ]
    },
    {
        'animal': 'Hippopotamus',
        'accepted_answers': ['Hippopotamus', 'Hippo', 'Common hippopotamus'],
        'identifying_features': [
            'is a barrel-shaped African river mammal that secretes a red blood sweat',
            'is a highly aggressive semi-aquatic mammal with enormous jaws and canine tusks'
        ],
        'jeopardy_questions': [
            'Which massive, semi-aquatic African mammal secretes a reddish natural sunscreen fluid often called "blood sweat"?',
            'What highly aggressive, barrel-shaped African river mammal has very short legs, enormous jaws, and large canine tusks?'
        ]
    },
    {
        'animal': 'Toucan',
        'accepted_answers': ['Toucan', 'Toco toucan'],
        'identifying_features': [
            'is a Neotropical bird with a giant, brightly colored beak',
            'has a disproportionately large bill supported inside by a lightweight bony honeycomb'
        ],
        'jeopardy_questions': [
            'Which tropical South American bird is famous for its enormous, brightly colored, yet surprisingly lightweight beak?',
            'What Neotropical bird possesses a giant, brightly colored bill supported inside by a lightweight bony honeycomb structure?'
        ]
    },
    {
        'animal': 'Skunk',
        'accepted_answers': ['Skunk', 'Skunks', 'Polecat', 'Mephitidae'],
        'identifying_features': [
            'is a black-and-white striped mammal famous for its foul-smelling spray',
            'raises its bushy tail to warn predators before spraying from its anal glands'
        ],
        'jeopardy_questions': [
            'Which black-and-white striped North American mammal is famous for spraying a foul-smelling liquid as a defense mechanism?',
            'What small mammalian omnivore raises its bushy tail to warn predators before spraying a highly pungent chemical from its anal glands?'
        ]
    },
    {
        'animal': 'Camel',
        'accepted_answers': ['Camel', 'Dromedary', 'Bactrian camel'],
        'identifying_features': [
            'is a large desert ungulate with prominent fatty humps on its back',
            'has thick eyelashes and sealable nostrils to survive harsh sandstorms'
        ],
        'jeopardy_questions': [
            'Which large, long-legged desert beast of burden is famous for having fatty humps on its back?',
            'What large desert ungulate has thick eyelashes, sealable nostrils, and humps on its back to survive harsh, dry environments?'
        ]
    },
    {
        'animal': 'Cheetah',
        'accepted_answers': ['Cheetah', 'Acinonyx jubatus'],
        'identifying_features': [
            'is a large spotted African cat famous for being the fastest land animal',
            'has distinctive black tear-like streaks running down its face'
        ],
        'jeopardy_questions': [
            'Which large spotted African wild cat is famous for being the fastest land animal in the world?',
            'What spotted wild cat has distinctive black tear-like streaks on its face and is renowned as the fastest land animal?'
        ]
    },
    {
        'animal': 'Peacock',
        'accepted_answers': ['Peacock', 'Peafowl', 'Pavo'],
        'identifying_features': [
            'is a brightly colored male bird that displays a massive, iridescent fan of tail feathers',
            'uses a train of eye-spotted feathers for elaborate courtship displays'
        ],
        'jeopardy_questions': [
            'Which specific male bird is widely famous for displaying a massive, iridescent fan of eye-spotted tail feathers?',
            'What specific brightly colored male bird shakes an enormous fan of tail feathers during elaborate courtship displays?'
        ]
    },
    {
        'animal': 'Porcupine',
        'accepted_answers': ['Porcupine'],
        'identifying_features': [
            'is a large, slow-moving rodent covered in sharp, detachable quills',
            'lodges its barbed defensive spines into the skin of attacking predators'
        ],
        'jeopardy_questions': [
            'Which large, slow-moving rodent defends itself with an array of sharp, detachable barbed quills?',
            'What large quill-bearing rodent can easily lodge its sharp, defensive spines into the skin of attacking predators?'
        ]
    },
    {
        'animal': 'Armadillo',
        'accepted_answers': ['Armadillo', 'Dasypodidae'],
        'identifying_features': [
            'is a New World mammal with a protective leathery armor shell made of overlapping bony plates',
            'rolls into a tight protective ball when threatened by predators'
        ],
        'jeopardy_questions': [
            'Which New World mammal has a protective leathery armor shell made of overlapping bony plates?',
            'What armored mammal from the Americas is known to roll into a tight protective ball when threatened by predators?'
        ]
    },
    {
        'animal': 'Wombat',
        'accepted_answers': ['Wombat', 'Common wombat', 'Vombatidae'],
        'identifying_features': [
            'is a stout, burrowing Australian marsupial uniquely famous for naturally producing cube-shaped feces',
            'has a backward-facing pouch to prevent dirt from entering while digging'
        ],
        'jeopardy_questions': [
            'Which stout, burrowing Australian marsupial (belonging to the family Vombatidae) is uniquely famous for naturally producing dry, cube-shaped feces?',
            'What stout, short-legged Australian marsupial has a backward-facing pouch to prevent dirt from entering while digging?'
        ]
    }
]



# 1. Create a copy of the original list of dictionaries
#results_list = copy.deepcopy(animals)


### Prev. Animals 